### In this notbook I train the mouse fullmodels needed for plotting the figures. I will save the models in a checkpoint folder (checkpoints_trained)

I reuse the notbook fullmodel_mouse and fittet it to my purposes

In [37]:
import os
import torch
import numpy as np
from minimodel import data

device = torch.device('cuda')

In [38]:
mouse_id = 5

data_path = '../data'
weight_path = './checkpoints_192-x'
results_path = './results_192-x'
os.makedirs(weight_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)
np.random.seed(1)

In [39]:
# load images
img = data.load_images(data_path, mouse_id, file=data.img_file_name[mouse_id])

raw image shape:  (68000, 66, 264)
cropped image shape:  (68000, 66, 130)
img:  (68000, 66, 130) -2.0829253 2.1060908 float32


In [40]:
# load neurons
fname = '%s_nat60k_%s.npz'%(data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(file_path = os.path.join(data_path, fname), mouse_id = mouse_id)
n_stim, n_neurons = spks.shape


loading activities from ../data/FX20_nat60k_2023_09_29.npz


In [41]:
# split train and validation set
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)


splitting training and validation set...
itrain:  (43081,)
ival:  (4787,)


In [42]:
# normalize data
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)


normalizing neural data...
finished


In [43]:
ineur = np.arange(0, n_neurons) #np.arange(0, n_neurons, 5)
spks_train = torch.from_numpy(spks[itrain][:,ineur]).to(device)
spks_val = torch.from_numpy(spks[ival][:,ineur]).to(device)

print('spks_train: ', spks_train.shape, spks_train.min(), spks_train.max())
print('spks_val: ', spks_val.shape, spks_val.min(), spks_val.max())

img_train = torch.from_numpy(img[istim_train][itrain]).to(device).unsqueeze(1) # change :130 to 25:100 
img_val = torch.from_numpy(img[istim_train][ival]).to(device).unsqueeze(1)
img_test = torch.from_numpy(img[istim_test]).to(device).unsqueeze(1)

print('img_train: ', img_train.shape, img_train.min(), img_train.max())
print('img_val: ', img_val.shape, img_val.min(), img_val.max())
print('img_test: ', img_test.shape, img_test.min(), img_test.max())

input_Ly, input_Lx = img_train.shape[-2:]

spks_train:  torch.Size([43081, 2746]) tensor(-1.4092e-15, device='cuda:0') tensor(48.7427, device='cuda:0')
spks_val:  torch.Size([4787, 2746]) tensor(-6.8745e-16, device='cuda:0') tensor(44.7361, device='cuda:0')
img_train:  torch.Size([43081, 1, 66, 130]) tensor(-2.0829, device='cuda:0') tensor(2.1061, device='cuda:0')
img_val:  torch.Size([4787, 1, 66, 130]) tensor(-2.0829, device='cuda:0') tensor(2.1061, device='cuda:0')
img_test:  torch.Size([500, 1, 66, 130]) tensor(-2.0829, device='cuda:0') tensor(2.1061, device='cuda:0')


In [44]:
from minimodel import model_builder
from minimodel import model_trainer
from minimodel import metrics

seed = 1
feve_nlayers = []
for nlayers in range(1, 5):
    # Building Model

    nconv1 = 192
    nconv2 = 192
    model, in_channels = model_builder.build_model(NN=len(ineur), n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2)
    model_name = model_builder.create_model_name(data.mouse_names[mouse_id], data.exp_date[mouse_id], n_layers=nlayers, in_channels=in_channels, seed=seed)
    
    model_path = os.path.join(weight_path, model_name)
    print('model path: ', model_path)
    model = model.to(device)


    # Training the model
    print(device)
    if not os.path.exists(model_path):
        best_state_dict = model_trainer.train(model, spks_train, spks_val, img_train, img_val, device=device)
        torch.save(best_state_dict, model_path)
        print('saved model', model_path)
    model.load_state_dict(torch.load(model_path))
    print('loaded model', model_path)

    # test model
    test_pred = model_trainer.test_epoch(model, img_test)
    print('test_pred: ', test_pred.shape, test_pred.min(), test_pred.max())


    test_fev, test_feve = metrics.feve(spks_rep_all, test_pred)
    print('FEVE (test, all): ', np.mean(test_feve))

    threshold = 0.15
    print(f'filtering neurons with FEV > {threshold}')
    valid_idxes = np.where(test_fev > threshold)[0]
    print(f'valid neurons: {len(valid_idxes)} / {len(test_fev)}')
    print(f'FEVE (test, FEV>0.15): {np.mean(test_feve[test_fev > threshold])}')

    feve_nlayers.append(np.mean(test_feve[test_fev > threshold]))

core shape:  torch.Size([1, 192, 33, 65])
input shape of readout:  (192, 33, 65)
model name:  FX20_092923_1layer_192_clamp_norm_depthsep_pool_xrange_176.pt
model path:  ./checkpoints_192-x/FX20_092923_1layer_192_clamp_norm_depthsep_pool_xrange_176.pt
cuda
loaded model ./checkpoints_192-x/FX20_092923_1layer_192_clamp_norm_depthsep_pool_xrange_176.pt
test_pred:  (500, 2746) 0.0029961467 7.2573676


/tmp/ipykernel_4018804/1921035201.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


FEVE (test, all):  2.0136704
filtering neurons with FEV > 0.15
valid neurons: 1239 / 2746
FEVE (test, FEV>0.15): 0.6027274131774902
core shape:  torch.Size([1, 192, 33, 65])
input shape of readout:  (192, 33, 65)
model name:  FX20_092923_2layer_192_192_clamp_norm_depthsep_pool_xrange_176.pt
model path:  ./checkpoints_192-x/FX20_092923_2layer_192_192_clamp_norm_depthsep_pool_xrange_176.pt
cuda
loaded model ./checkpoints_192-x/FX20_092923_2layer_192_192_clamp_norm_depthsep_pool_xrange_176.pt
test_pred:  (500, 2746) 0.001691401 8.554103
FEVE (test, all):  2.0446942
filtering neurons with FEV > 0.15
valid neurons: 1239 / 2746
FEVE (test, FEV>0.15): 0.7071778774261475
core shape:  torch.Size([1, 192, 33, 65])
input shape of readout:  (192, 33, 65)
model name:  FX20_092923_3layer_192_192_192_clamp_norm_depthsep_pool_xrange_176.pt
model path:  ./checkpoints_192-x/FX20_092923_3layer_192_192_192_clamp_norm_depthsep_pool_xrange_176.pt
cuda
loaded model ./checkpoints_192-x/FX20_092923_3layer_192_

In [45]:

# ---- Saving performance scores ----
file_name = "results_" + str(mouse_id)
results_file_path = os.path.join(results_path, file_name)

feve_nlayers = np.array(feve_nlayers)
print("saving array of shape: ", feve_nlayers.shape)
np.savez(results_file_path, FEVE_scores=feve_nlayers)
print(f"Results saved at: {results_file_path}")

saving array of shape:  (4,)
Results saved at: ./results_192-x/results_5
